# Тестирование RAG Pipeline в Google Colab

Этот ноутбук предназначен для быстрого тестирования проекта в Google Colab.

**Важно:** Включите GPU в настройках Colab (Runtime → Change runtime type → GPU)


## 1. Клонирование репозитория


In [ ]:
!git clone https://github.com/phantom2059/baseline_rag.git
!cd baseline_rag && pwd


## 2. Установка зависимостей


In [ ]:
import sys
sys.path.insert(0, '/content/baseline_rag')

%cd baseline_rag
%pip install -q -U pip
%pip install -q -r requirements.txt

# Для Colab используем faiss-cpu (faiss-gpu может не работать)
%pip install -q faiss-gpu

print("✓ Зависимости установлены")


## 3. Проверка GPU


In [ ]:
import torch

print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠ GPU не обнаружен. Включите GPU в настройках Colab!")


## 4. Создание тестовых данных


In [ ]:
import json
from pathlib import Path

# Создаём тестовый файл с более сложными вопросами для демонстрации RAG
test_questions = [
    "Объясни разницу между градиентным спуском и стохастическим градиентным спуском",
    "Какие основные принципы работы трансформеров в архитектуре BERT?",
    "Что такое attention mechanism и как он используется в нейросетях?",
    "Опиши процесс обучения нейронной сети с обратным распространением ошибки",
    "Какие преимущества и недостатки у рекуррентных нейронных сетей по сравнению с трансформерами?",
    "Что такое dropout и зачем он нужен при обучении глубоких сетей?",
    "Объясни концепцию transfer learning в контексте компьютерного зрения",
    "Как работает механизм квантования моделей и какие типы квантования существуют?",
    "Что такое RAG и как он улучшает качество ответов языковых моделей?",
    "Опиши процесс создания векторных эмбеддингов для текстовых данных"
]

input_file = Path("/content/baseline_rag/input.json")
with open(input_file, "w", encoding="utf-8") as f:
    json.dump(test_questions, f, ensure_ascii=False, indent=2)

print(f"✓ Создан файл с {len(test_questions)} сложными вопросами")
print("Примеры вопросов:")
for i, q in enumerate(test_questions[:3], 1):
    print(f"  {i}. {q[:70]}..." if len(q) > 70 else f"  {i}. {q}")


## 5. Загрузка модели (опционально)

**Внимание:** Скачивание модели может занять 5-10 минут и потребует ~15GB дискового пространства.

Если модель уже скачана в предыдущей сессии, установите `DOWNLOAD_MODEL = False`


In [ ]:
import sys
sys.path.insert(0, '/content/baseline_rag')

from model_downloader import download_model

DOWNLOAD_MODEL = True  # Поставьте False, если модель уже скачана
MODEL_DIR = "/content/baseline_rag/model"

if DOWNLOAD_MODEL:
    print("[INFO] Начинаем скачивание модели...")
    print("[INFO] Это может занять 5-10 минут...")
    download_model(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        model_dir=MODEL_DIR,
        quantize=False,
        #bits=8,  # 8-bit для экономии VRAM в Colab
        config_path="/content/baseline_rag/config.json",
    )
    print("✓ Модель скачана и сохранена")
else:
    print("[SKIP] Используем существующую модель")


## 6. Загрузка вопросов и инференс


In [ ]:
import sys
sys.path.insert(0, '/content/baseline_rag')

from utils import load_questions
from factual_model import FactualModel

# Загружаем вопросы
questions = load_questions("/content/baseline_rag/input.json")
print(f"✓ Загружено {len(questions)} вопросов")

# Инициализируем модель
print("\n[INFO] Инициализация модели (это может занять 1-2 минуты)...")
model = FactualModel(config_path="/content/baseline_rag/config.json")
print("✓ Модель готова")

# Генерируем ответы
print("\n[INFO] Генерация ответов...")
answers = model.generate_batch(questions, batch_size=8)  # Меньший batch для Colab
print(f"✓ Получено {len(answers)} ответов")


## 7. Вывод результатов


In [ ]:
from tabulate import tabulate

rows = []
for idx, (q, a) in enumerate(zip(questions, answers), start=1):
    rows.append({
        "#": idx,
        "Вопрос": q if len(q) <= 60 else q[:57] + "...",
        "Ответ": a,
    })

print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ:")
print("="*80)
print(tabulate(rows, headers="keys", tablefmt="grid"))


## 8. Сохранение результатов


In [ ]:
import sys
sys.path.insert(0, '/content/baseline_rag')

from utils import save_results

output_path = "/content/baseline_rag/output_colab.json"
save_results(questions, answers, output_path, fmt="json")

print(f"✓ Результаты сохранены в {output_path}")
print("\nВы можете скачать файл через меню Files → output_colab.json")


## Дополнительно: Проверка использования памяти
 

In [ ]:
import torch

if torch.cuda.is_available():
    print("Использование GPU:")
    print(f"  Выделено: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  Кэшировано: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"  Всего доступно: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("GPU не используется")
